# Ihyun — project lead notebook

**Project:** Does a structure predictor notice a mutation?
**Team:** CAIR Structural Biology / Molecular Biophysics, Fall 2026
**Role:** folding runs (GPU), controls, statistics, scientific framing

---

This notebook holds the work only the lead can do: panel selection, the substitution
assertion over the whole panel, folding dispatch, the noise floor, and the paired
statistics. Member-facing utilities get promoted into `src/` as soon as a second person
needs them.

See `../README.md` for the question and the hypotheses.

## Setup

In [ ]:
import sys
sys.path.append('../src')

from mutations import parse_variant, apply_variant, assert_variant_set, infer_offset
from geometry import load_ca, superpose, locality_profile, noise_floor

# Small-matrix numpy work: pin BLAS to one thread.
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')


---

## Week 6 — panel frozen, data verified

Three things have to be true before anything is folded:

1. The DMS assay exists and is downloadable for every protein on the panel.
2. The numbering offset between the DMS table and the sequence we fold is **known**,
   not guessed, and written into `data/README.md`.
3. `assert_variant_set` passes on the whole panel.

Step 3 is the one that silently ruins a quarter if skipped. See `src/mutations.py`.

In [ ]:
# Panel. Lengths and assay availability are UNVERIFIED background knowledge —
# confirming them is this week's deliverable. Do not fold anything until they are.
PANEL = {
    "ubiquitin":  {"length": 76,  "dms": None, "offset": None},
    "psd95_pdz3": {"length": 83,  "dms": None, "offset": None},
    "avgfp":      {"length": 238, "dms": None, "offset": None},
    "tem1":       {"length": 286, "dms": None, "offset": None},  # stretch goal
}
PANEL

In [ ]:
# The assertion, run over a real panel. This is the week 6 gate.
seq_ubq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
variants = ["K48R", "L8A", "I44A", "K63R"]

summary = assert_variant_set(seq_ubq, variants, offset=0)
print(summary)

# If everything fails, it is almost always a constant offset, not bad data:
# infer_offset(seq_ubq, variants)

### Sampling design

~100 substitutions per protein, stratified on two axes:

| axis | strata |
|---|---|
| measured fitness | neutral vs. strongly deleterious |
| burial | buried vs. solvent-exposed |

Random sampling would only support reporting an average. Stratifying is what makes
*"does the response depend on structural context?"* answerable, and it is free at
selection time and impossible to retrofit.

---

## Week 7 — the noise floor

**Fold the same wild-type sequence several times under identical settings**, then take
every pairwise RMSD. That distribution is the threshold any mutant effect has to clear.

Without this number, a mutant RMSD of 0.3 Å is uninterpretable. The papers claiming
AlphaFold is insensitive to mutations do not appear to report one, which is part of why
this is worth doing.

In [ ]:
# reps = [load_ca(f"../data/processed/ubq_wt_rep{i}.pdb") for i in range(1, 6)]
# floor = noise_floor(reps)
# floor

### Hoffman2 notes

- **Public/shared queue only.** Do not pass `-l highp` — that allocation belongs to the
  Kappel Lab and this is CAIR work.
- Never pass `-q` on a GPU job; request the GPU through `-l`. A queue list plus a generic
  `-l gpu` sits in `qw` forever with no error.
- `cuda=N` is the consumable the scheduler actually waits on, not CPU slots.
- An empty `qstat` means "nothing queued", not "nothing new to fetch". Pull results
  explicitly.
- Keep login nodes to metadata. Importing torch on one has gotten the session killed.

---

## Week 8 — ESMFold mutants, geometry end to end

In [ ]:
# wt  = load_ca("../data/processed/ubq_wt.pdb")
# mut = load_ca("../data/processed/ubq_K48R.pdb")
# d = superpose(wt, mut)
# print(d.rmsd, d.max_displacement, d.mean_delta_plddt)
# locality_profile(d, wt, site=48)

---

## Week 9 — AlphaFold2 arm, paired comparison, stratification

**Open decision to settle before this runs:** does the ColabFold arm rebuild the MSA per
mutant, or reuse the wild-type alignment? Reusing is standard practice and is arguably
the very thing under criticism; rebuilding is more honest and slower. Whichever we pick,
document it here — a reviewer will ask.

In [ ]:
# paired comparison: ESMFold vs AlphaFold2 displacement, same substitutions


---

## Week 10 — figures and the talk

Lead with the noise floor and the paired comparison. The fitness-correlation arm is
optional and is the one most likely to reproduce a known negative.

---


## Literature review

> **Format note:** the CAIR handbook defines an official literature review format that
> was not included in the brief handed to this team. Replace this template with the
> official one once you have it. Until then, this covers the same ground.

### Paper

**Citation:**
**Link / DOI:**
**Provenance:** full-text / metadata-only / background-knowledge *(pick one — see below)*

> **Provenance matters on this team.** Label every source as **full-text** (you read the
> paper), **metadata-only** (title and abstract only), or **background-knowledge** (you
> are recalling it, not checking it). This is how we avoid asserting things in a talk
> that nobody actually verified. Our whole project premise is currently metadata-only.

### What question were they asking?

### What did they do?

_Data, method, and the comparison that carries the claim._

### What did they find?

_Numbers, not impressions. If you cannot give a number, say so._

### What is the weakest part?

_Every paper has one. Sample size, a missing control, a benchmark that leaks. For our
project specifically: did they report a run-to-run noise floor? Most do not._

### What does this mean for us?

_One or two sentences. Does it support H1, H0, or neither? Does it change what we
should measure?_

### Questions I still have
